# FIFA World Cup 2026 — Monte Carlo Predictor

**Model stack:**
- Current Elo ratings (computed from 150 years of results)
- Recent attack / defence ratings (last 4 years of competitive matches)
- Dixon-Coles Poisson correction (adjusts for observed excess of 0-0 and 1-1 draws)
- 10 000 full-tournament Monte Carlo simulations

**Tournament format (48 teams):**
- 12 groups of 4 → top 2 + best 8 third-place teams = 32 knockout teams
- Round of 32 → Round of 16 → Quarter-finals → Semi-finals → Final

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from collections import defaultdict
import unicodedata, warnings
warnings.filterwarnings('ignore')

np.random.seed(2026)

plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor':   '#161b22',
    'axes.edgecolor':   '#30363d',
    'text.color':       '#e6edf3',
    'axes.labelcolor':  '#e6edf3',
    'xtick.color':      '#8b949e',
    'ytick.color':      '#8b949e',
    'grid.color':       '#21262d',
    'grid.alpha':       0.5,
})

def normalise(s):
    """Normalise unicode so team names match across CSV files."""
    return unicodedata.normalize('NFC', str(s).strip())

## 1. Load Data

In [ ]:
results      = pd.read_csv('../data/raw/results.csv', parse_dates=['date'])
elo_df       = pd.read_csv('../data/processed/current_elo_ratings.csv')
team_matches = pd.read_csv('../data/processed/team_matches.csv', parse_dates=['date'])

# Normalise unicode in team names throughout
for df in [results, team_matches]:
    for col in ['home_team','away_team'] if 'home_team' in df.columns else ['team','opponent']:
        df[col] = df[col].map(normalise)
elo_df['team'] = elo_df['team'].map(normalise)

elo_map = dict(zip(elo_df['team'], elo_df['current_elo']))

print(f"Historical matches : {len(results):,}")
print(f"Teams with Elo     : {len(elo_map)}")
print(f"Team-match rows    : {len(team_matches):,}")

## 2. World Cup 2026 Groups (auto-extracted from results.csv)

In [ ]:
wc26 = results[
    (results['tournament'] == 'FIFA World Cup') &
    (results['date'] >= '2026-06-01') &
    (results['home_score'].isna())
].copy()

# Recover groups via connected-component analysis:
# each team plays every other team in its group exactly once
adjacency = defaultdict(set)
for _, row in wc26.iterrows():
    adjacency[row['home_team']].add(row['away_team'])
    adjacency[row['away_team']].add(row['home_team'])

visited, groups = set(), []
for team in sorted(adjacency):
    if team not in visited:
        component, stack = [], [team]
        while stack:
            t = stack.pop()
            if t not in visited:
                visited.add(t)
                component.append(t)
                stack.extend(adjacency[t] - visited)
        groups.append(sorted(component))

groups.sort(key=lambda g: g[0])
GROUP_LABELS = 'ABCDEFGHIJKL'
GROUP_MAP = {team: GROUP_LABELS[i] for i, g in enumerate(groups) for team in g}

print("=== FIFA World Cup 2026 Groups ===")
for i, g in enumerate(groups):
    print(f"  Group {GROUP_LABELS[i]}: {', '.join(g)}")

ALL_TEAMS = [t for g in groups for t in g]
print(f"\nTotal qualified teams : {len(ALL_TEAMS)}")

## 3. Team Strength — Elo + Recent Attack / Defence

In [ ]:
CUTOFF       = pd.Timestamp('2026-06-01')
RECENT_START = CUTOFF - pd.DateOffset(years=4)

# Exclude low-stakes friendlies from the ratings
SKIP = {'Friendly', 'Kirin Cup', 'Intercontinental Cup'}

competitive = results[
    (results['date'] >= RECENT_START) &
    (results['date'] < CUTOFF) &
    results['home_score'].notna() &
    ~results['tournament'].isin(SKIP)
].copy()

def team_stats(df):
    rows = []
    for _, r in df.iterrows():
        rows.append({'team': r['home_team'], 'gf': r['home_score'], 'ga': r['away_score']})
        rows.append({'team': r['away_team'], 'gf': r['away_score'], 'ga': r['home_score']})
    t = pd.DataFrame(rows)
    return t.groupby('team').agg(gf_mean=('gf','mean'), ga_mean=('ga','mean'), n=('gf','count'))

recent_stats   = team_stats(competitive)
league_avg_gf  = recent_stats['gf_mean'].mean()
league_avg_ga  = recent_stats['ga_mean'].mean()

recent_stats['attack']  = recent_stats['gf_mean'] / league_avg_gf
recent_stats['defence'] = recent_stats['ga_mean'] / league_avg_ga   # lower = better

wc_elos = {t: elo_map.get(t, 1500) for t in ALL_TEAMS}
max_elo = max(wc_elos.values())

records = []
for team in ALL_TEAMS:
    elo = wc_elos[team]
    # Use data-derived stats if >= 5 competitive matches, else fall back to Elo proxy
    elo_frac = (elo - 1300) / (max_elo - 1300)   # 0..1
    if team in recent_stats.index and recent_stats.loc[team, 'n'] >= 5:
        atk = recent_stats.loc[team, 'attack']
        dfn = recent_stats.loc[team, 'defence']
        n   = int(recent_stats.loc[team, 'n'])
    else:
        atk = 0.5 + elo_frac * 1.0   # range ~0.5 – 1.5
        dfn = 1.5 - elo_frac * 1.0   # range ~0.5 – 1.5 (lower = better)
        n   = 0
    records.append({
        'team': team, 'elo': elo,
        'attack': atk, 'defence': dfn,
        'group': GROUP_MAP[team], 'n_recent': n
    })

strength = pd.DataFrame(records).set_index('team')

WC_BASE        = 1.15   # WC average goals per team per game
GLOBAL_ATK_AVG = strength['attack'].mean()
GLOBAL_DFN_AVG = strength['defence'].mean()

print(f"Avg attack index : {GLOBAL_ATK_AVG:.3f}  |  Avg defence index : {GLOBAL_DFN_AVG:.3f}")
print(f"\nTop 12 by Elo:")
print(strength.sort_values('elo', ascending=False)[['elo','attack','defence','n_recent']].head(12).to_string())

## 4. Dixon-Coles Poisson Match Simulation

In [ ]:
RHO = -0.13   # Dixon-Coles correction for low-scoring scorelines

def dc_correction(ga, gb, mu_a, mu_b, rho=RHO):
    if   ga == 0 and gb == 0: return max(1 - mu_a * mu_b * rho, 0.01)
    elif ga == 0 and gb == 1: return max(1 + mu_a * rho, 0.01)
    elif ga == 1 and gb == 0: return max(1 + mu_b * rho, 0.01)
    elif ga == 1 and gb == 1: return max(1 - rho, 0.01)
    return 1.0

def expected_goals(team_a, team_b):
    """Expected goals for a neutral-venue WC match."""
    s_a = strength.loc[team_a]
    s_b = strength.loc[team_b]
    mu_a = WC_BASE * (s_a['attack'] / GLOBAL_ATK_AVG) * (GLOBAL_DFN_AVG / max(s_b['defence'], 0.3))
    mu_b = WC_BASE * (s_b['attack'] / GLOBAL_ATK_AVG) * (GLOBAL_DFN_AVG / max(s_a['defence'], 0.3))
    # Soft Elo blend: adjusts mu by up to ±15%
    elo_prob_a = 1 / (1 + 10 ** ((s_b['elo'] - s_a['elo']) / 400))
    elo_adj    = (elo_prob_a - 0.5) * 0.30
    mu_a = max(mu_a * (1 + elo_adj), 0.1)
    mu_b = max(mu_b * (1 - elo_adj), 0.1)
    return mu_a, mu_b

def simulate_match(team_a, team_b, knockout=False):
    """
    Simulate one match using Dixon-Coles Poisson.
    In knockout mode a draw goes to a penalty shootout (Elo-weighted coin flip).
    Returns (goals_a, goals_b, winner_or_None).
    """
    mu_a, mu_b = expected_goals(team_a, team_b)
    # DC rejection-sampling (usually accepts on first draw)
    for _ in range(20):
        ga = np.random.poisson(mu_a)
        gb = np.random.poisson(mu_b)
        if np.random.random() < dc_correction(ga, gb, mu_a, mu_b):
            break
    if knockout and ga == gb:
        p_pen = 1 / (1 + 10 ** ((strength.loc[team_b,'elo'] - strength.loc[team_a,'elo']) / 800))
        winner = team_a if np.random.random() < p_pen else team_b
    else:
        winner = team_a if ga > gb else (team_b if gb > ga else None)
    return ga, gb, winner

# Sanity checks
for a, b in [('Argentina', 'Algeria'), ('Spain', 'France'), ('Germany', 'Curaçao')]:
    if a in strength.index and b in strength.index:
        mu_a, mu_b = expected_goals(a, b)
        print(f"{a} vs {b}:  {mu_a:.2f} – {mu_b:.2f}")

## 5. Group Stage Simulator

In [ ]:
def simulate_group(teams):
    """Round-robin for 4 teams. Returns standings sorted by pts → gd → gf."""
    stats = {t: {'pts':0,'gf':0,'ga':0,'gd':0} for t in teams}
    for i in range(len(teams)):
        for j in range(i+1, len(teams)):
            a, b = teams[i], teams[j]
            ga, gb, _ = simulate_match(a, b, knockout=False)
            stats[a]['gf'] += ga;  stats[a]['ga'] += gb;  stats[a]['gd'] += ga - gb
            stats[b]['gf'] += gb;  stats[b]['ga'] += ga;  stats[b]['gd'] += gb - ga
            if ga > gb:   stats[a]['pts'] += 3
            elif gb > ga: stats[b]['pts'] += 3
            else:         stats[a]['pts'] += 1; stats[b]['pts'] += 1
    df = pd.DataFrame(stats).T
    df = df.sort_values(['pts','gd','gf'], ascending=False)
    return df

def best_third_teams(third_records):
    """Pick best 8 of 12 third-place teams by pts → gd → gf."""
    df = pd.DataFrame(third_records).sort_values(['pts','gd','gf'], ascending=False)
    return df.iloc[:8]['team'].tolist()

# Quick group test
print("Test group (Group A — Argentina et al.):")
print(simulate_group(groups[0]))

## 6. Knockout Bracket Builder

In [ ]:
def build_r32_bracket(group_results, qual_thirds):
    """
    Build 16 R32 matchups from 24 group qualifiers + 8 best third-place teams.

    Pairing logic (mirrors FIFA 2026 bracket structure):
      - Groups A-H: winner[i] vs runner[i^1] (cross-pair adjacent groups)
      - Groups I-L: winners and runners vs the 8 qualified third-place teams
    """
    winners  = [group_results[GROUP_LABELS[i]][0] for i in range(12)]
    runners  = [group_results[GROUP_LABELS[i]][1] for i in range(12)]
    thirds   = sorted(qual_thirds, key=lambda t: -strength.loc[t,'elo'])

    bracket = []
    # Cross-pair first 8 groups: 1A-2B, 1B-2A, 1C-2D, 1D-2C, 1E-2F, 1F-2E, 1G-2H, 1H-2G
    cross = [(0,1),(1,0),(2,3),(3,2),(4,5),(5,4),(6,7),(7,6)]
    for wi, ri in cross:
        bracket.append((winners[wi], runners[ri]))

    # Remaining 4 winners (I-L) and 4 runners (I-L) each face a third-place team
    for i in range(4):
        bracket.append((winners[8+i], thirds[i]))
    for i in range(4):
        bracket.append((runners[8+i], thirds[4+i]))

    assert len(bracket) == 16, f"Expected 16 R32 matches, got {len(bracket)}"
    all_teams = [t for pair in bracket for t in pair]
    assert len(all_teams) == len(set(all_teams)), "Duplicate team in R32 bracket!"
    return bracket

def play_round(pairs):
    """Simulate one knockout round; return list of winners (preserving bracket order)."""
    return [simulate_match(a, b, knockout=True)[2] for a, b in pairs]

print("Bracket builder ready.")

## 7. Full Tournament Simulation

In [ ]:
ROUNDS = ['Group', 'R32', 'R16', 'QF', 'SF', 'Runner-up', 'Winner']

def simulate_tournament():
    """
    Simulate one full WC2026 tournament.
    Returns dict {team: furthest_round_reached}.
    'Group' = eliminated in group stage.
    """
    reached = {t: 'Group' for t in ALL_TEAMS}

    # ── Group stage ──────────────────────────────────────────────────────────
    group_results  = {}   # label → (winner, runner_up)
    third_records  = []

    for i, g in enumerate(groups):
        label     = GROUP_LABELS[i]
        standings = simulate_group(g)
        w  = standings.index[0]
        ru = standings.index[1]
        th = standings.index[2]
        group_results[label] = (w, ru)
        reached[w]  = 'R32'
        reached[ru] = 'R32'
        third_records.append({
            'team': th,
            'pts':  int(standings.loc[th, 'pts']),
            'gd':   int(standings.loc[th, 'gd']),
            'gf':   int(standings.loc[th, 'gf']),
        })

    qual_thirds = best_third_teams(third_records)
    for t in qual_thirds:
        reached[t] = 'R32'

    # ── Knockout rounds ───────────────────────────────────────────────────────
    r32_bracket = build_r32_bracket(group_results, qual_thirds)

    r16_teams = play_round(r32_bracket)
    for t in r16_teams: reached[t] = 'R16'

    r16_pairs = list(zip(r16_teams[::2], r16_teams[1::2]))
    qf_teams  = play_round(r16_pairs)
    for t in qf_teams: reached[t] = 'QF'

    qf_pairs = list(zip(qf_teams[::2], qf_teams[1::2]))
    sf_teams = play_round(qf_pairs)
    for t in sf_teams: reached[t] = 'SF'

    sf_pairs   = list(zip(sf_teams[::2], sf_teams[1::2]))
    finalists  = play_round(sf_pairs)
    for t in finalists: reached[t] = 'Runner-up'

    _, _, champion = simulate_match(finalists[0], finalists[1], knockout=True)
    reached[champion] = 'Winner'

    return reached

# Smoke test
trial  = simulate_tournament()
winner = [t for t, r in trial.items() if r == 'Winner'][0]
print("Smoke test winner:", winner)
for rnd in ROUNDS:
    n = sum(1 for r in trial.values() if r == rnd)
    print(f"  {rnd:12s}: {n} teams")

## 8. Monte Carlo — 10 000 Simulations

In [ ]:
N_SIMS = 10_000

# counts[team][round] = number of sims where team reached AT LEAST that round
counts = {t: defaultdict(int) for t in ALL_TEAMS}

print(f"Running {N_SIMS:,} simulations...")
for sim in range(N_SIMS):
    if (sim + 1) % 2000 == 0:
        print(f"  {sim+1:,} / {N_SIMS:,}")
    result = simulate_tournament()
    for team, round_reached in result.items():
        # Credit every round FROM the start UP TO the round reached
        # e.g. if team reaches 'QF', they also get credit for Group, R32, R16, QF
        idx = ROUNDS.index(round_reached)
        for r in ROUNDS[:idx + 1]:
            counts[team][r] += 1

# Build probability DataFrame
probs = pd.DataFrame({
    team: {r: counts[team][r] / N_SIMS for r in ROUNDS}
    for team in ALL_TEAMS
}).T
probs.index.name = 'team'
probs['group'] = probs.index.map(GROUP_MAP)
probs['elo']   = probs.index.map(lambda t: strength.loc[t, 'elo'])
probs = probs.sort_values('Winner', ascending=False)

print("\nTop 15 — Win probability:")
print(probs[['group','elo','Winner','Runner-up','SF','QF','R16','R32']]
      .head(15).to_string(float_format='{:.1%}'.format))

## 9. Visualisations

In [ ]:
# ── Confederation colour map ─────────────────────────────────────────────────
CONF_COLORS = {
    'UEFA':     '#58a6ff',
    'CONMEBOL': '#3fb950',
    'AFC':      '#d2a8ff',
    'CAF':      '#ffa657',
    'CONCACAF': '#f78166',
    'OFC':      '#79c0ff',
}

CONFEDERATION = {
    'Spain':'UEFA','France':'UEFA','England':'UEFA','Germany':'UEFA','Portugal':'UEFA',
    'Netherlands':'UEFA','Belgium':'UEFA','Croatia':'UEFA','Denmark':'UEFA',
    'Switzerland':'UEFA','Austria':'UEFA','Turkey':'UEFA','Serbia':'UEFA',
    'Sweden':'UEFA','Norway':'UEFA','Czech Republic':'UEFA',
    'Bosnia and Herzegovina':'UEFA','Scotland':'UEFA','Hungary':'UEFA',
    'Argentina':'CONMEBOL','Brazil':'CONMEBOL','Colombia':'CONMEBOL',
    'Uruguay':'CONMEBOL','Ecuador':'CONMEBOL','Paraguay':'CONMEBOL',
    'Chile':'CONMEBOL','Venezuela':'CONMEBOL',
    'Algeria':'CAF','Morocco':'CAF','Senegal':'CAF','Egypt':'CAF',
    'Ivory Coast':'CAF','South Africa':'CAF','Ghana':'CAF',
    'DR Congo':'CAF','Tunisia':'CAF','Cameroon':'CAF','Cape Verde':'CAF',
    'Japan':'AFC','South Korea':'AFC','Iran':'AFC','Saudi Arabia':'AFC',
    'Australia':'AFC','Iraq':'AFC','Jordan':'AFC','Uzbekistan':'AFC',
    'Qatar':'AFC','Bahrain':'AFC',
    'United States':'CONCACAF','Mexico':'CONCACAF','Canada':'CONCACAF',
    'Panama':'CONCACAF','Honduras':'CONCACAF','Jamaica':'CONCACAF',
    'Haiti':'CONCACAF','Costa Rica':'CONCACAF','Curaçao':'CONCACAF',
    'New Zealand':'OFC',
}

def conf_color(team):
    return CONF_COLORS.get(CONFEDERATION.get(normalise(team), 'UEFA'), '#8b949e')

print("Colour map ready.")

In [ ]:
# ── 9a. Win Probability — Top 24 ────────────────────────────────────────────
top24  = probs.head(24).copy()
colors = [conf_color(t) for t in top24.index]

fig, ax = plt.subplots(figsize=(14, 9))
bars = ax.barh(range(len(top24)), top24['Winner'] * 100,
               color=colors, edgecolor='#21262d', linewidth=0.5, height=0.75)

for bar, (team, row) in zip(bars, top24.iterrows()):
    w = bar.get_width()
    ax.text(w + 0.2, bar.get_y() + bar.get_height()/2,
            f"{row['Winner']*100:.1f}%   Elo {row['elo']:.0f}",
            va='center', ha='left', fontsize=8.5, color='#8b949e')

ax.set_yticks(range(len(top24)))
ax.set_yticklabels([f"{t}  [{GROUP_MAP.get(t,'?')}]" for t in top24.index], fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Championship Probability (%)', fontsize=10)
ax.set_title(f'FIFA World Cup 2026 — Win Probability\n({N_SIMS:,} Monte Carlo simulations)',
             fontsize=13, pad=12)
ax.grid(axis='x', alpha=0.3)
ax.set_xlim(0, top24['Winner'].max() * 100 * 1.4)

legend_patches = [mpatches.Patch(color=c, label=k) for k, c in CONF_COLORS.items()]
ax.legend(handles=legend_patches, loc='lower right', fontsize=8,
          facecolor='#161b22', edgecolor='#30363d')

plt.tight_layout()
plt.savefig('../models/wc2026_win_probability.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved → models/wc2026_win_probability.png")

In [ ]:
# ── 9b. Advancement Heatmap — Top 32 ────────────────────────────────────────
top32        = probs.head(32).copy()
heat_cols    = ['R32', 'R16', 'QF', 'SF', 'Runner-up', 'Winner']
heat_data    = top32[heat_cols].values * 100

fig, ax = plt.subplots(figsize=(11, 14))
im = ax.imshow(heat_data, aspect='auto', cmap='YlOrRd', vmin=0, vmax=100)

ax.set_xticks(range(len(heat_cols)))
ax.set_xticklabels(heat_cols, fontsize=10)
ax.set_yticks(range(len(top32)))
ax.set_yticklabels(
    [f"{i+1:2d}. {t}  [{GROUP_MAP.get(t,'?')}]" for i, t in enumerate(top32.index)],
    fontsize=8.5)

for i in range(len(top32)):
    for j in range(len(heat_cols)):
        val = heat_data[i, j]
        ax.text(j, i, f"{val:.0f}%",
                ha='center', va='center', fontsize=7,
                color='black' if val > 55 else '#e6edf3', fontweight='bold')

plt.colorbar(im, ax=ax, label='Probability (%)', shrink=0.35)
ax.set_title(f'FIFA World Cup 2026 — Advancement Probabilities\n({N_SIMS:,} simulations)',
             fontsize=12, pad=10)
plt.tight_layout()
plt.savefig('../models/wc2026_advancement_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved → models/wc2026_advancement_heatmap.png")

In [ ]:
# ── 9c. Group-by-Group Breakdown ─────────────────────────────────────────────
from matplotlib.patches import Patch

fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.flatten()

for idx, (i, g) in enumerate(enumerate(groups)):
    label = GROUP_LABELS[i]
    ax    = axes[idx]
    gp    = probs.loc[g].sort_values('Winner', ascending=False)
    clrs  = [conf_color(t) for t in gp.index]
    x     = range(4)

    # Stacked: Win | Finalist extra | SF extra
    win_pct  = gp['Winner']   * 100
    fin_pct  = (gp['Runner-up'] - gp['Winner'])   * 100
    sf_pct   = (gp['SF']        - gp['Runner-up']) * 100

    ax.bar(x, win_pct, color=clrs, alpha=0.95)
    ax.bar(x, fin_pct, bottom=win_pct, color=clrs, alpha=0.55, hatch='//')
    ax.bar(x, sf_pct,  bottom=win_pct + fin_pct, color=clrs, alpha=0.30, hatch='..')

    ax.set_title(f'Group {label}', fontsize=10, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels([t.replace(' ','\n') for t in gp.index], fontsize=7.5)
    ax.set_ylabel('%' if idx % 4 == 0 else '', fontsize=8)
    ax.set_ylim(0, 80)
    ax.grid(axis='y', alpha=0.3)

    for xi, team in zip(x, gp.index):
        elo = strength.loc[team, 'elo']
        ax.text(xi, 1, f"{elo:.0f}", ha='center', va='bottom', fontsize=6.5, color='#8b949e')

legend_el = [
    Patch(facecolor='grey', alpha=0.95, label='Win %'),
    Patch(facecolor='grey', alpha=0.55, hatch='//', label='Finalist %'),
    Patch(facecolor='grey', alpha=0.30, hatch='..', label='Semi-final %'),
]
fig.legend(handles=legend_el, loc='lower center', ncol=3, fontsize=9,
           facecolor='#161b22', edgecolor='#30363d', bbox_to_anchor=(0.5, 0.01))
fig.suptitle(f'FIFA World Cup 2026 — Group Breakdown  ({N_SIMS:,} simulations)',
             fontsize=13, y=1.01)
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig('../models/wc2026_groups.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved → models/wc2026_groups.png")

In [ ]:
# ── 9d. Attack vs Defence Scatter ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 8))

for team in ALL_TEAMS:
    s       = strength.loc[team]
    win_pct = probs.loc[team, 'Winner'] * 100
    size    = 40 + win_pct * 30
    ax.scatter(s['attack'], s['defence'], color=conf_color(team),
               s=size, alpha=0.8, edgecolors='#30363d', linewidth=0.5)
    if win_pct >= 1.5:
        ax.annotate(team, (s['attack'], s['defence']),
                    fontsize=7, color='#e6edf3', alpha=0.9,
                    xytext=(5, 4), textcoords='offset points')

ax.axhline(GLOBAL_DFN_AVG, color='#30363d', linestyle='--', alpha=0.6)
ax.axvline(GLOBAL_ATK_AVG, color='#30363d', linestyle='--', alpha=0.6)
ax.set_xlabel('Attack Index  (higher → more goals scored)',  fontsize=10)
ax.set_ylabel('Defence Index  (lower → fewer goals conceded)', fontsize=10)
ax.set_title('FIFA World Cup 2026 — Team Strength Profile\n'
             'Attack vs Defence  ·  bubble size ∝ win probability', fontsize=11)

legend_patches = [mpatches.Patch(color=c, label=k) for k, c in CONF_COLORS.items()]
ax.legend(handles=legend_patches, loc='upper right', fontsize=8,
          facecolor='#161b22', edgecolor='#30363d')
plt.tight_layout()
plt.savefig('../models/wc2026_strength_profile.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved → models/wc2026_strength_profile.png")

In [ ]:
# ── 9e. Full Predictions Table ───────────────────────────────────────────────
summary = probs[['group','elo','R32','R16','QF','SF','Runner-up','Winner']].copy()
summary.columns = ['Grp','Elo','R32%','R16%','QF%','SF%','Final%','Win%']
for col in ['R32%','R16%','QF%','SF%','Final%','Win%']:
    summary[col] = (summary[col] * 100).round(1)
summary['Elo'] = summary['Elo'].round(0).astype(int)
summary = summary.reset_index()

print("=" * 82)
print(f"FIFA WORLD CUP 2026 — PREDICTIONS  ({N_SIMS:,} Monte Carlo simulations)")
print("=" * 82)
print(summary.to_string(index=False))

import os
os.makedirs('../models', exist_ok=True)
summary.to_csv('../models/wc2026_predictions.csv', index=False)
print("\nSaved → models/wc2026_predictions.csv")

## 10. Re-run with More Simulations

Change `N_SIMS = 10_000` in cell 8 to any value and re-run cells 8–9.  
Standard error on win % ≈ `sqrt(p*(1-p)/N)` — at 10 000 sims it's ±0.5% for a 5% favourite.